<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/Leonardo-branch/Trabalho1_Aquisicao_Dados_Estiagem_AM_Final_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho 1 — Aquisição de Dados
## Estiagem 2023/2024 no Amazonas × Indicadores Econômicos (IBGE)

**Disciplina:** Ciência de Dados — Instituto de Computação (UFAM)
(Aquisição de Dados)

**Data da coleta:** gerada automaticamente — ver coluna `data_hora_coleta_utc` em `registro_proveniencia.csv` (Seção 8)

---


### Fontes utilizadas

| # | Fonte | Método | O que fornece |
|---|-------|--------|----------------|
| 1 | **Boletins/notícias sobre a estiagem 2023/2024** (Defesa Civil AM via imprensa) | **Web scraping** (`requests` + `BeautifulSoup`) | Nº de municípios em emergência, pessoas/famílias afetadas e cota do Rio Negro, por data de boletim |
| 2 | **API do IBGE** (Localidades + SIDRA/Agregados) | **API REST** | Lista dos 62 municípios do AM (código e nome); **IPCA mensal** e **taxa de desocupação** (PNAD Contínua) |

**Chave de integração:** `ano_mes` (`AAAA-MM`). A granularidade diária dos boletins é convertida
para mensal antes do merge (detalhes na Seção 6).

### Decisões metodológicas importantes (leia antes de rodar)

- **IPCA e Manaus.** O IPCA do IBGE **não tem Manaus como área de coleta** (as áreas são RMs de
  Belém, Fortaleza, Recife, Salvador, BH, Vitória, RJ, SP, Curitiba e Porto Alegre, mais Brasília,
  Goiânia, Campo Grande, Rio Branco, São Luís e Aracaju). O notebook **procura Manaus na tabela**
  e, se não existir, usa um **proxy regional** (Belém, depois Rio Branco, depois Brasil). A
  localidade realmente usada fica registrada na coluna `ipca_localidade` da base.
- **Períodos explícitos.** Os períodos são calculados a partir de `JANELA_INICIO`/`JANELA_FIM`
  (Seção 1). **Não** usamos o atalho relativo `-N` da API, que devolve "os últimos N períodos" na
  data de execução e deixaria a coleta fora da janela da estiagem.
- **Desocupação trimestral.** A PNAD Contínua trimestral tem códigos `AAAA01`–`AAAA04`
  (trimestres), que **não** são meses. O código converte cada trimestre nos seus 3 meses
  (o valor do trimestre é repetido nos meses) e guarda o período original em
  `desocupacao_periodo_original`.
- **Valores manuais têm prioridade sobre os automáticos** (o regex sobre texto jornalístico pode
  errar), e cada valor tem uma coluna `origem_*` indicando de onde veio. Divergências entre
  automático e manual são listadas na Seção 6.1.

### Observação sobre robustez

Páginas de notícias mudam de estrutura com o tempo. As células de scraping são defensivas
(try/except, regex tolerante, validação de faixa de valores), **gravam o HTML bruto (bytes originais)
antes de qualquer extração** e registram o hash SHA-256 no log de proveniência. Se algum site
retornar erro (403, timeout), é necessario ajuste a URL ou use a conferência manual (Seção 3).

## 1. Setup do ambiente

In [ ]:
# Bibliotecas — no Google Colab, requests/bs4/pandas/lxml já vêm instaladas.
import os
import re
import json
import time
import hashlib
import zipfile
import datetime as dt
import urllib.robotparser as robotparser
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    from unidecode import unidecode
except ImportError:
    %pip install -q unidecode
    from unidecode import unidecode

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Ambiente pronto.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.0 MB/s eta 0:00:00
Ambiente pronto.


In [ ]:
import pathlib
from google.colab import userdata

USUARIO = "PedroPontoPNG"
REPO    = "crise-saude-ufam"
'''
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/{USUARIO}/{REPO}.git
%cd {REPO}
!git config user.name "Pedro Montenegro"
!git config user.email "pedro.montenegro@icomp.ufam.edu.br"

print("Repositório clonado em:", pathlib.Path.cwd())

NOME_BRANCH = "pedro/trabalho1-estiagem-am"

!git checkout -b {NOME_BRANCH}
print("Branch criada e ativa:", NOME_BRANCH)


In [ ]:
# ======================= CONFIGURAÇÃO (edite aqui) =======================
CONTATO_EMAIL = " "
USER_AGENT_NOME = "TesteAcademico"

# Janela temporal da análise (inclusive). Cobre antes/durante/depois da estiagem de 2023.
JANELA_INICIO = "2022-09"
JANELA_FIM = "2024-12"

# Regra (ARBITRÁRIA, documentada) para o rótulo "em_crise": >= N municípios em emergência.
LIMIAR_CRISE_MUNICIPIOS = 50

DELAY_ENTRE_REQUISICOES = 2            # segundos
ROBOTS_INDETERMINADO_PROSSEGUIR = False  # se robots.txt não puder ser lido: True = raspa mesmo assim

# IDs de tabelas do SIDRA (conferidos por nome/metadados na Seção 4; troque aqui se necessário)
ID_TABELA_IPCA = 7060           # IPCA — variação mensal etc. (a partir de jan/2020)
ID_TABELA_DESOCUPACAO = 4099    # PNAD Contínua trimestral — taxa de desocupação (14 anos ou mais)

# Colab tem disco efêmero: para não perder tudo ao desconectar, monte o Drive.
MONTAR_DRIVE = False
INCLUIR_HTML_BRUTO_NO_ZIP = False   # HTML de matérias é conteúdo protegido: por padrão NÃO vai no zip

# ==========================================================================
BASE_DIR = "."
if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/trabalho1_estiagem"

DIR_BRUTOS = os.path.join(BASE_DIR, "dados_brutos")
DIR_BRUTOS_SCRAPING = os.path.join(DIR_BRUTOS, "scraping_estiagem")
DIR_BRUTOS_IBGE = os.path.join(DIR_BRUTOS, "api_ibge")
DIR_TRATADOS = os.path.join(BASE_DIR, "dados_tratados")

for d in [DIR_BRUTOS, DIR_BRUTOS_SCRAPING, DIR_BRUTOS_IBGE, DIR_TRATADOS]:
    os.makedirs(d, exist_ok=True)

HEADERS = {
    "User-Agent": (
        f"Mozilla/5.0 (compatible; {USER_AGENT_NOME}/1.1; "
        f"uso academico, sem fins comerciais; contato: {CONTATO_EMAIL})"
    )
}
if "SEU_EMAIL_AQUI" in CONTATO_EMAIL:
    print("[aviso] Preencha CONTATO_EMAIL: o User-Agent deve permitir que os sites contatem os autores.")

MESES_JANELA = list(pd.period_range(JANELA_INICIO, JANELA_FIM, freq="M").strftime("%Y-%m"))
print("Pastas:", DIR_BRUTOS, "|", DIR_TRATADOS)
print(f"Janela de análise: {MESES_JANELA[0]} a {MESES_JANELA[-1]} ({len(MESES_JANELA)} meses)")

Pastas: ./dados_brutos | ./dados_tratados
Janela de análise: 2022-09 a 2024-12 (28 meses)


In [ ]:
# --- Utilitários gerais: proveniência, hash, HTTP com retry, números pt-BR ----------
provenance_log = []


def agora_utc():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")


def sha256_bytes(conteudo: bytes) -> str:
    return hashlib.sha256(conteudo).hexdigest()


def registrar_provenancia(fonte, url, metodo, parametros=None, observacao="", arquivo=None, sha256=None):
    """Anota fonte, URL exata, data/hora, método, parâmetros e (se houver) arquivo bruto + hash."""
    registro = {
        "fonte": fonte,
        "url": url,
        "data_hora_coleta_utc": agora_utc(),
        "metodo": metodo,
        "parametros": json.dumps(parametros or {}, ensure_ascii=False),
        "observacao": observacao,
        "arquivo_bruto": arquivo,
        "sha256": sha256,
    }
    provenance_log.append(registro)
    return registro


def salvar_bruto(caminho, conteudo: bytes) -> str:
    """Grava os bytes EXATOS recebidos e devolve o SHA-256."""
    with open(caminho, "wb") as f:
        f.write(conteudo)
    return sha256_bytes(conteudo)


def salvar_json_bruto(caminho, obj) -> str:
    return salvar_bruto(caminho, json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))


def http_get(url, tentativas=3, timeout=30, espera=2, **kwargs):
    """GET com retry/backoff para erros de rede e HTTP 429/5xx. Pode devolver resposta 4xx."""
    ultimo_erro = None
    for i in range(1, tentativas + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout, **kwargs)
            if resp.status_code in (429, 500, 502, 503, 504) and i < tentativas:
                time.sleep(espera * i)
                continue
            return resp
        except requests.exceptions.RequestException as e:
            ultimo_erro = e
            if i < tentativas:
                time.sleep(espera * i)
    raise ultimo_erro


def parse_num_ptbr(txt):
    """Converte número em texto para float, tratando pt-BR ('1.234,5') e ponto decimal ('12.89').
    - '12,89' -> 12.89 | '12.89' -> 12.89 | '1.289' -> 1289 (milhar) | '1.234,5' -> 1234.5"""
    s = str(txt).strip()
    if not s:
        return None
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    elif "." in s and re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    try:
        return float(s)
    except ValueError:
        return None


print("Utilitários prontos.")

Utilitários prontos.


## 2. Verificação de `robots.txt`

Antes de raspar, checamos o `robots.txt` **de cada URL que será acessada** (não só da raiz do
domínio), usando nosso próprio User-Agent. Resultado possível:

- `permitido` — pode raspar;
- `bloqueado` — a URL é **pulada** pelo laço de scraping;
- `indeterminado` — não foi possível ler o `robots.txt` (erro de rede, 401/403, 5xx). Por
  padrão (postura conservadora) a URL também é **pulada**

Se o `robots.txt` declarar `Crawl-delay`, o maior entre ele e `DELAY_ENTRE_REQUISICOES` é usado.
Cada leitura de `robots.txt` entra no log de proveniência.

In [ ]:
_robots_cache = {}


def _carregar_robots(url_alvo):
    p = urlparse(url_alvo)
    base = f"{p.scheme}://{p.netloc}"
    if base in _robots_cache:
        return _robots_cache[base]

    robots_url = f"{base}/robots.txt"
    info = {"robots_url": robots_url, "parser": None, "status": "indeterminado", "detalhe": ""}
    try:
        resp = http_get(robots_url, tentativas=2, timeout=15)
        if resp.status_code == 200:
            rp = robotparser.RobotFileParser()
            rp.parse(resp.text.splitlines())
            info.update(parser=rp, status="ok", detalhe="HTTP 200")
        elif resp.status_code in (404, 410):
            info.update(status="sem_robots", detalhe=f"HTTP {resp.status_code}: sem robots.txt")
        else:  # 401/403/429/5xx...: não dá para saber a política do site
            info.update(status="indeterminado", detalhe=f"HTTP {resp.status_code}")
    except Exception as e:
        info["detalhe"] = f"erro de rede: {e}"

    registrar_provenancia(
        fonte=f"robots.txt de {p.netloc}", url=robots_url,
        metodo="GET (requests + urllib.robotparser)", parametros={"user_agent": USER_AGENT_NOME},
        observacao=f"status={info['status']}; {info['detalhe']}",
    )
    _robots_cache[base] = info
    return info


def checar_robots(url_alvo):
    """Retorna dict com 'pode' ('permitido'|'bloqueado'|'indeterminado'), 'crawl_delay' e detalhes."""
    info = _carregar_robots(url_alvo)
    delay = None
    if info["status"] == "ok":
        rp = info["parser"]
        pode = "permitido" if rp.can_fetch(USER_AGENT_NOME, url_alvo) else "bloqueado"
        delay = rp.crawl_delay(USER_AGENT_NOME)
    elif info["status"] == "sem_robots":
        pode = "permitido"
    else:
        pode = "indeterminado"
    return {"pode": pode, "crawl_delay": delay, "robots_url": info["robots_url"], "detalhe": info["detalhe"]}


print("Checagem de robots.txt pronta.")

Checagem de robots.txt pronta.


## 3. Fonte 1 — Web Scraping: boletins da estiagem 2023/2024 no Amazonas

A Defesa Civil do Amazonas divulgou boletins periódicos durante a estiagem de 2023 (e novamente em
2024) informando: número de municípios em situação de emergência, número de pessoas e de
famílias afetadas e, em vários boletins, a cota (nível, em metros) do Rio Negro em Manaus. Esses
boletins não têm API pública, mas foram republicados em páginas HTML de portais de notícia, que é
o que raspamos aqui.

> **Limitação conhecida:** cada matéria cobre uma data específica; a série resultante é **esparsa**
> (poucas datas, com um buraco entre nov/2023 e jul/2024). Também há republicações da mesma notícia
> (o IHU/ClimaInfo republica matérias de outros veículos), então as linhas **não são observações
> independentes**. Ver Seção 11 (próximos passos).

**Confira cada URL manualmente** (título, data, teor) antes de usar; ajuste/substitua livremente por
fontes equivalentes, de preferência **primárias** (Defesa Civil AM, Governo do Amazonas, SGB/CPRM).

In [ ]:
# Lista curada de páginas com boletins/números da estiagem 2023-2024.
# data_ref = data de referência atribuída pelo grupo (será comparada com a data publicada na página).
URLS_ESTIAGEM = [
    {"data_ref": "2023-09-29", "fonte": "Poder360",
     "url": "https://www.poder360.com.br/brasil/governo-do-am-decreta-emergencia-em-55-cidades-por-seca-severa/"},
    {"data_ref": "2023-10-19", "fonte": "IHU/ClimaInfo",
     "url": "https://ihu.unisinos.br/633364-amazonas-tem-a-menor-superficie-de-agua-desde-2018-mostra-mapbiomas"},
    {"data_ref": "2023-10-23", "fonte": "Jornal de Brasília",
     "url": "https://jornaldebrasilia.com.br/noticias/brasil/seca-no-amazonas-afeta-mais-de-630-mil-pessoas/"},
    {"data_ref": "2023-10-25", "fonte": "IHU/ClimaInfo",
     "url": "https://ihu.unisinos.br/633559-numero-de-afetados-pela-seca-severa-no-amazonas-sobe-para-633-mil-pess"},
    {"data_ref": "2023-10-27", "fonte": "Diário de Pernambuco",
     "url": "https://www.diariodepernambuco.com.br/noticia/brasil/2023/10/seca-afeta-todos-os-62-municipios-do-amazonas.html"},
    # DUPLICATA DE DATA (2023-10-27) COMENTADA: a chave de junção é `data_ref`, então duas URLs no
    # mesmo dia colidiam no merge. Mantivemos a Diário de Pernambuco (bate com o valor manual de
    # 62 municípios); a Rede Onda Digital cobria o mesmo fato e foi removida da coleta automática.
    # {"data_ref": "2023-10-27", "fonte": "Rede Onda Digital",
    #  "url": "https://redeondadigital.com.br/amazonas/seca-todos-municipios-am-agora-afetados"},
    {"data_ref": "2023-10-28", "fonte": "IHU/ClimaInfo",
     "url": "https://ihu.unisinos.br/633708-quase-todos-os-municipios-do-am-estao-em-situacao-de-emergencia-por-causa-da-seca"},
    {"data_ref": "2023-11-19", "fonte": "Rede Onda Digital",
     "url": "https://redeondadigital.com.br/amazonas/estiagem-150-mil-familias-amazonas"},
    {"data_ref": "2024-07-12", "fonte": "IHU/ClimaInfo",
     "url": "https://ihu.unisinos.br/641315-seca-no-amazonas-chega-antes-do-previsto-e-cidades-entram-em-emergencia"},
    # ATENÇÃO: esta URL tem o padrão '/categorias/...', diferente das demais — confira se abre a matéria.
    {"data_ref": "2024-08-23", "fonte": "IHU/ClimaInfo",
     "url": "https://ihu.unisinos.br/categorias/642732-rios-amazonicos-registram-baixa-historica-um-mes-antes-do-pico-da-estiagem"},
]

df_urls = pd.DataFrame(URLS_ESTIAGEM)
df_urls

,data_ref,fonte,url
0,2023-09-29,Poder360,https://www.poder360.com.br/brasil/governo-do-...
1,2023-10-19,IHU/ClimaInfo,https://ihu.unisinos.br/633364-amazonas-tem-a-...
2,2023-10-23,Jornal de Brasília,https://jornaldebrasilia.com.br/noticias/brasi...
3,2023-10-25,IHU/ClimaInfo,https://ihu.unisinos.br/633559-numero-de-afeta...
4,2023-10-27,Diário de Pernambuco,https://www.diariodepernambuco.com.br/noticia/...
5,2023-10-28,IHU/ClimaInfo,https://ihu.unisinos.br/633708-quase-todos-os-...
6,2023-11-19,Rede Onda Digital,https://redeondadigital.com.br/amazonas/estiag...
7,2024-07-12,IHU/ClimaInfo,https://ihu.unisinos.br/641315-seca-no-amazona...
8,2024-08-23,IHU/ClimaInfo,https://ihu.unisinos.br/categorias/642732-rios...


In [ ]:
# Checagem de robots.txt de CADA URL (com o nosso User-Agent). O resultado é usado pelo laço de scraping.
robots_resultado = []
for item in URLS_ESTIAGEM:
    r = checar_robots(item["url"])
    item["robots"] = r["pode"]
    item["crawl_delay"] = r["crawl_delay"]
    robots_resultado.append({
        "dominio": urlparse(item["url"]).netloc, "url": item["url"], "robots_url": r["robots_url"],
        "resultado": r["pode"], "crawl_delay": r["crawl_delay"], "detalhe": r["detalhe"],
    })

df_robots = pd.DataFrame(robots_resultado)
print(df_robots["resultado"].value_counts().to_string())
df_robots

resultado
permitido    9


,dominio,url,robots_url,resultado,crawl_delay,detalhe
0,www.poder360.com.br,https://www.poder360.com.br/brasil/governo-do-...,https://www.poder360.com.br/robots.txt,permitido,None,HTTP 200
1,ihu.unisinos.br,https://ihu.unisinos.br/633364-amazonas-tem-a-...,https://ihu.unisinos.br/robots.txt,permitido,None,HTTP 200
2,jornaldebrasilia.com.br,https://jornaldebrasilia.com.br/noticias/brasi...,https://jornaldebrasilia.com.br/robots.txt,permitido,None,HTTP 200
3,ihu.unisinos.br,https://ihu.unisinos.br/633559-numero-de-afeta...,https://ihu.unisinos.br/robots.txt,permitido,None,HTTP 200
4,www.diariodepernambuco.com.br,https://www.diariodepernambuco.com.br/noticia/...,https://www.diariodepernambuco.com.br/robots.txt,permitido,None,HTTP 200
5,ihu.unisinos.br,https://ihu.unisinos.br/633708-quase-todos-os-...,https://ihu.unisinos.br/robots.txt,permitido,None,HTTP 200
6,redeondadigital.com.br,https://redeondadigital.com.br/amazonas/estiag...,https://redeondadigital.com.br/robots.txt,permitido,None,HTTP 200
7,ihu.unisinos.br,https://ihu.unisinos.br/641315-seca-no-amazona...,https://ihu.unisinos.br/robots.txt,permitido,None,HTTP 200
8,ihu.unisinos.br,https://ihu.unisinos.br/categorias/642732-rios...,https://ihu.unisinos.br/robots.txt,permitido,None,HTTP 200


**Leitura do resultado acima:** URLs `bloqueado` são puladas. URLs `indeterminado` também são puladas
(a menos que `ROBOTS_INDETERMINADO_PROSSEGUIR = True`). Erros 401/403 na leitura do `robots.txt`
costumam vir de proteção anti-bot (ex.: Cloudflare), não de uma política do site — nesses casos,
use a conferência manual (abaixo) ou decida conscientemente prosseguir e **registre a decisão**.

In [ ]:
def extrair_texto_pagina(html_bytes):
    """Devolve (titulo, data_publicacao_meta, texto_do_conteudo_principal).
    Foca no <article>/<main> e remove blocos de ruído (relacionadas, comentários, compartilhar...)
    para não capturar números de OUTRAS matérias listadas na página."""
    def _sopa():
        return BeautifulSoup(html_bytes, "lxml")

    soup = _sopa()
    titulo = soup.title.get_text(strip=True) if soup.title else None

    data_meta = None
    for attrs in ({"property": "article:published_time"}, {"name": "article:published_time"},
                  {"itemprop": "datePublished"}, {"name": "date"}, {"property": "og:updated_time"}):
        tag = soup.find("meta", attrs=attrs)
        if tag is not None and tag.get("content"):
            data_meta = tag["content"][:10]
            break
    if data_meta is None:
        t = soup.find("time", attrs={"datetime": True})
        if t is not None:
            data_meta = t["datetime"][:10]

    # Limpeza principal (NÃO remove <header>: o lead/título da matéria costuma estar nele)
    for tag in soup(["script", "style", "noscript", "nav", "footer", "aside", "form", "iframe"]):
        tag.decompose()
    ruido = re.compile(
        r"relacionad|leia-?mais|leia-?tamb|related|comment|coment|share|compartilh|"
        r"newsletter|sidebar|widget|publicidade|advert", re.I)
    for tag in soup.find_all(attrs={"class": ruido}) + soup.find_all(attrs={"id": ruido}):
        if not getattr(tag, "decomposed", False):
            tag.decompose()

    area = soup.find("article") or soup.find("main") or soup.body or soup
    texto = re.sub(r"\s+", " ", area.get_text(separator=" ")).strip()

    # Se a limpeza por classe apagou quase tudo (classe de "ruído" em um contêiner grande), refaz
    # SEM o filtro por classe/id, mas ainda sem menu/rodapé/lateral, e avisa no texto de retorno.
    if len(texto) < 300:
        soup2 = _sopa()
        for tag in soup2(["script", "style", "noscript", "nav", "footer", "aside", "form", "iframe"]):
            tag.decompose()
        estreito = re.compile(r"relacionad|related|leia-?mais|leia-?tamb", re.I)
        for tag in soup2.find_all(attrs={"class": estreito}) + soup2.find_all(attrs={"id": estreito}):
            if not getattr(tag, "decomposed", False):
                tag.decompose()
        area2 = soup2.find("article") or soup2.find("main") or soup2.body or soup2
        texto = re.sub(r"\s+", " ", area2.get_text(separator=" ")).strip()

    return titulo, data_meta, texto


# Padrões (regex) para os números dos boletins. Cada item: (regex, divisor). O grupo 1 é o número.
# Valores em "mil" -> divisor 1; valores absolutos ("608.432 pessoas") -> divisor 1000 (colunas em milhares).
_MUN = r"munic[íi]pios|cidades"
PADROES = {
    "municipios_emergencia": [
        (re.compile(r"emerg[êe]ncia\s+em\s+(\d{1,2})\s+(?:" + _MUN + r")", re.I), 1),
        (re.compile(r"(\d{1,2})\s+(?:" + _MUN + r")[^.]{0,60}?emerg[êe]ncia", re.I), 1),
    ],
    "pessoas_afetadas_mil": [
        (re.compile(r"(\d{1,3}(?:[.,]\d{1,3})?)\s*mil\s+pessoas", re.I), 1),
        (re.compile(r"(\d{1,3}(?:\.\d{3})+)\s+pessoas", re.I), 1000),
    ],
    "familias_afetadas_mil": [
        (re.compile(r"(\d{1,3}(?:[.,]\d{1,3})?)\s*mil\s+fam[íi]lias", re.I), 1),
        (re.compile(r"(\d{1,3}(?:\.\d{3})+)\s+fam[íi]lias", re.I), 1000),
    ],
    "cota_rio_negro_m": [
        (re.compile(r"rio\s+negro[^.]{0,120}?(\d{1,2}[.,]\d{1,2})\s*(?:metros|m\b)", re.I), 1),
        (re.compile(r"(\d{1,2}[.,]\d{1,2})\s*(?:metros|m\b)[^.]{0,80}?rio\s+negro", re.I), 1),
    ],
}

# Faixas plausíveis: valor fora da faixa é descartado (evita capturar números de outro contexto).
# NOTA: 'das 62 cidades do estado' é o TOTAL de municípios, não os em emergência — por isso NÃO há
# mais o fallback 'total_municipios_afetados' que existia na versão anterior.
FAIXAS = {
    "municipios_emergencia": (1, 62),
    "pessoas_afetadas_mil": (1, 5000),
    "familias_afetadas_mil": (1, 1500),
    "cota_rio_negro_m": (5, 35),
}


def extrair_campo(campo, texto):
    """Primeiro casamento válido (dentro da faixa). Devolve (valor, trecho_de_evidência)."""
    lo, hi = FAIXAS[campo]
    for regex, divisor in PADROES[campo]:
        for m in regex.finditer(texto):
            v = parse_num_ptbr(m.group(1))
            if v is None:
                continue
            v = v / divisor
            if lo <= v <= hi:
                return v, texto[max(0, m.start() - 60): m.end() + 60]
    return None, None


print("Extratores prontos.")

Extratores prontos.


In [ ]:
registros_scraping = []

for item in URLS_ESTIAGEM:
    url = item["url"]
    dominio = urlparse(url).netloc
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", url.rstrip("/").split("/")[-1])[:80]
    caminho_bruto = os.path.join(DIR_BRUTOS_SCRAPING, f"{item['data_ref']}_{slug}.html")

    # Respeita robots.txt (Seção 2)
    if item["robots"] == "bloqueado" or (item["robots"] == "indeterminado" and not ROBOTS_INDETERMINADO_PROSSEGUIR):
        print(f"PULADO (robots={item['robots']}) {item['data_ref']} {dominio}")
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="(não coletado)", parametros={},
            observacao=f"URL pulada: robots.txt = {item['robots']}",
        )
        continue

    try:
        resp = http_get(url, tentativas=2, timeout=20)
        resp.raise_for_status()

        # 1) PRESERVAÇÃO DO BRUTO: bytes exatamente como vieram, antes de qualquer parsing.
        sha = salvar_bruto(caminho_bruto, resp.content)
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="GET (requests + BeautifulSoup)",
            parametros={"headers": "User-Agent customizado", "timeout": 20},
            observacao=f"HTTP {resp.status_code}", arquivo=caminho_bruto, sha256=sha,
        )

        # 2) Extração (BeautifulSoup recebe bytes e detecta o encoding sozinho)
        titulo, data_meta, texto = extrair_texto_pagina(resp.content)
        registro = {
            "data_ref": item["data_ref"], "fonte": item["fonte"], "url": url, "titulo": titulo,
            "data_publicacao_meta": data_meta,
            "data_divergente": bool(data_meta and data_meta[:7] != item["data_ref"][:7]),
            "arquivo_bruto": caminho_bruto,
        }
        evidencias = {}
        for campo in PADROES:
            valor, trecho = extrair_campo(campo, texto)
            registro[campo] = valor
            if trecho:
                evidencias[campo] = trecho
        registro["evidencias"] = json.dumps(evidencias, ensure_ascii=False)
        registros_scraping.append(registro)
        print(f"OK   {item['data_ref']}  {dominio}")

    except Exception as e:
        print(f"FALHOU {item['data_ref']} {dominio}: {e}")
        registrar_provenancia(
            fonte=item["fonte"], url=url, metodo="GET (requests)", parametros={}, observacao=f"ERRO: {e}",
        )

    time.sleep(max(DELAY_ENTRE_REQUISICOES, item.get("crawl_delay") or 0))

df_scraping_raw = pd.DataFrame(registros_scraping)
if len(df_scraping_raw):
    n_div = int(df_scraping_raw["data_divergente"].sum())
    if n_div:
        print(f"\n[atenção] {n_div} página(s) com data publicada em mês diferente de data_ref — confira `data_publicacao_meta`.")
df_scraping_raw

FALHOU 2023-09-29 www.poder360.com.br: 403 Client Error: Forbidden for url: https://www.poder360.com.br/brasil/governo-do-am-decreta-emergencia-em-55-cidades-por-seca-severa/
OK   2023-10-19  ihu.unisinos.br
OK   2023-10-23  jornaldebrasilia.com.br
OK   2023-10-25  ihu.unisinos.br
OK   2023-10-27  www.diariodepernambuco.com.br
OK   2023-10-28  ihu.unisinos.br
FALHOU 2023-11-19 redeondadigital.com.br: 403 Client Error: Forbidden for url: https://redeondadigital.com.br/amazonas/estiagem-150-mil-familias-amazonas
OK   2024-07-12  ihu.unisinos.br
OK   2024-08-23  ihu.unisinos.br

[atenção] 1 página(s) com data publicada em mês diferente de data_ref — confira `data_publicacao_meta`.


,data_ref,fonte,url,titulo,data_publicacao_meta,data_divergente,arquivo_bruto,municipios_emergencia,pessoas_afetadas_mil,familias_afetadas_mil,cota_rio_negro_m,evidencias
0,2023-10-19,IHU/ClimaInfo,https://ihu.unisinos.br/633364-amazonas-tem-a-...,Amazonas tem a menor superfície de água desde ...,None,False,./dados_brutos/scraping_estiagem/2023-10-19_63...,NaN,NaN,NaN,NaN,{}
1,2023-10-23,Jornal de Brasília,https://jornaldebrasilia.com.br/noticias/brasi...,Seca no Amazonas afeta mais de 630 mil pessoas...,None,False,./dados_brutos/scraping_estiagem/2023-10-23_se...,62.0,630.0,158.0,12.89,"{""municipios_emergencia"": ""ela Defesa Civil do..."
2,2023-10-25,IHU/ClimaInfo,https://ihu.unisinos.br/633559-numero-de-afeta...,Número de afetados pela seca severa no Amazona...,None,False,./dados_brutos/scraping_estiagem/2023-10-25_63...,62.0,633.0,158.0,NaN,"{""municipios_emergencia"": ""undo a Defesa Civil..."
3,2023-10-27,Diário de Pernambuco,https://www.diariodepernambuco.com.br/noticia/...,Seca afeta todos os 62 municípios do Amazonas ...,2025-09-02,True,./dados_brutos/scraping_estiagem/2023-10-27_se...,62.0,608.0,152.0,12.70,"{""municipios_emergencia"": ""das pela seca deste..."
4,2023-10-28,IHU/ClimaInfo,https://ihu.unisinos.br/633708-quase-todos-os-...,Quase todos os municípios do AM estão em situa...,None,False,./dados_brutos/scraping_estiagem/2023-10-28_63...,60.0,600.0,NaN,NaN,"{""municipios_emergencia"": ""egião amazônica . A..."
5,2024-07-12,IHU/ClimaInfo,https://ihu.unisinos.br/641315-seca-no-amazona...,Seca no Amazonas chega antes do previsto e cid...,None,False,./dados_brutos/scraping_estiagem/2024-07-12_64...,20.0,NaN,NaN,26.49,"{""municipios_emergencia"": ""ira passada (5/7), ..."
6,2024-08-23,IHU/ClimaInfo,https://ihu.unisinos.br/categorias/642732-rios...,Rios amazônicos registram baixa histórica um m...,None,False,./dados_brutos/scraping_estiagem/2024-08-23_64...,53.0,NaN,NaN,NaN,"{""municipios_emergencia"": ""a Civil de Amazonas..."


### Conferência manual (prioridade sobre o automático)

Como a extração automática depende de regex sobre texto jornalístico (formato não padronizado),
é normal que alguns campos venham vazios — nem toda matéria cita todos os números — ou errados.
A tabela abaixo é uma **conferência manual** das mesmas páginas de `URLS_ESTIAGEM`. Na Seção 6.1
ela tem **prioridade** sobre o valor automático, e toda divergência é listada.

> ⚠️ **O grupo deve rever cada linha abaixo contra a página original antes de entregar.** Esses
> valores foram carregados da versão anterior do notebook e não foram reverificados nesta revisão.

In [ ]:
dados_manuais_conferencia = [
    {"data_ref": "2023-09-29", "municipios_emergencia": 55, "pessoas_afetadas_mil": None,  "familias_afetadas_mil": None,  "cota_rio_negro_m": None},
    # familias_afetadas_mil removida (era um None que não fazia diferença nenhuma no resultado, já
    # que combine_first já cai para o automático quando o manual é nulo — mas ficava como um "falso
    # None" sugerindo uma checagem manual que não existiu; removida por clareza) -> o valor de
    # 158 mil vem direto da extração automática, que acertou este campo.
    {"data_ref": "2023-10-23", "municipios_emergencia": 59, "pessoas_afetadas_mil": 630.0, "cota_rio_negro_m": 12.89},
    {"data_ref": "2023-10-25", "municipios_emergencia": 59, "pessoas_afetadas_mil": 633.0, "familias_afetadas_mil": 158.0, "cota_rio_negro_m": None},
    # municipios_emergencia corrigido de 60 para 62: no dia 27/10/2023 a seca já afetava TODOS os
    # 62 municípios do Amazonas (ver título da matéria do Diário de Pernambuco em URLS_ESTIAGEM).
    {"data_ref": "2023-10-27", "municipios_emergencia": 62, "pessoas_afetadas_mil": 608.0, "familias_afetadas_mil": 152.0, "cota_rio_negro_m": None},
    {"data_ref": "2023-11-19", "municipios_emergencia": 62, "pessoas_afetadas_mil": 598.0, "familias_afetadas_mil": 150.0, "cota_rio_negro_m": None},
    # cota_rio_negro_m removida (mesmo motivo do familias_afetadas_mil acima) -> o valor de 26,49 m
    # vem direto da extração automática, que acertou este campo.
    {"data_ref": "2024-07-12", "municipios_emergencia": 20, "pessoas_afetadas_mil": None,  "familias_afetadas_mil": None},
    # municipios_emergencia corrigido de 20 para 53: a linha anterior repetia por engano o valor de
    # 2024-07-12 (erro de copia-e-cola); 23/08/2024 teve um número diferente de municípios.
    {"data_ref": "2024-08-23", "municipios_emergencia": 53, "pessoas_afetadas_mil": None,  "familias_afetadas_mil": None,  "cota_rio_negro_m": None},
]
df_manual = pd.DataFrame(dados_manuais_conferencia)
for c in ["municipios_emergencia", "pessoas_afetadas_mil", "familias_afetadas_mil", "cota_rio_negro_m"]:
    df_manual[c] = df_manual[c].astype(float)

caminho_manual = os.path.join(DIR_BRUTOS_SCRAPING, "conferencia_manual_boletins.csv")
df_manual.to_csv(caminho_manual, index=False)
with open(caminho_manual, "rb") as f:
    _sha_manual = sha256_bytes(f.read())
registrar_provenancia(
    fonte="Conferência manual (mesmas URLs de URLS_ESTIAGEM)",
    url="ver coluna url em URLS_ESTIAGEM",
    metodo="Leitura manual pelos autores do trabalho",
    parametros={}, observacao="valores digitados pelos autores; revisar contra as páginas",
    arquivo=caminho_manual, sha256=_sha_manual,
)
df_manual

,data_ref,municipios_emergencia,pessoas_afetadas_mil,familias_afetadas_mil,cota_rio_negro_m
0,2023-09-29,55.0,NaN,NaN,NaN
1,2023-10-23,59.0,630.0,NaN,12.89
2,2023-10-25,59.0,633.0,158.0,NaN
3,2023-10-27,62.0,608.0,152.0,NaN
4,2023-11-19,62.0,598.0,150.0,NaN
5,2024-07-12,20.0,NaN,NaN,NaN
6,2024-08-23,53.0,NaN,NaN,NaN


## 4. Fonte 2 — API do IBGE

Duas famílias de chamadas:

1. **Localidades** — lista oficial dos 62 municípios do Amazonas (código IBGE e nome). É uma
   dimensão de apoio para as fases futuras (agrupamento/geolocalização). *Não* traz população.
2. **SIDRA/Agregados** — indicadores econômicos: **IPCA mensal** (tabela 7060) e **taxa de
   desocupação** da PNAD Contínua (tabela 4099, trimestral, nível UF).

**Como os parâmetros são definidos (mudanças em relação à versão anterior):**

- **IDs de tabela fixos** (Seção 1), conferidos em tempo de execução pelo nome e pelos metadados
  impressos abaixo — em vez de pegar o "primeiro resultado" de uma busca no catálogo, cuja ordem é arbitrária.
- **Localidade descoberta pelo nome** via `/agregados/{id}/localidades/{nivel}` (nada de código
  hardcoded): para o IPCA procura Manaus e, se não existir, usa o proxy regional; para a
  desocupação procura o Amazonas (UF) e, se faltar, a Região Norte.
- **Períodos explícitos**: lista os períodos reais da tabela (`/agregados/{id}/periodos`) e
  seleciona os que intersectam a janela `JANELA_INICIO`–`JANELA_FIM`. A frequência (mensal ou
  trimestral) é inferida dos próprios códigos de período.

In [ ]:
URL_MUNICIPIOS_AM = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/AM/municipios"

resp = http_get(URL_MUNICIPIOS_AM, timeout=20)
resp.raise_for_status()
municipios_am_raw = resp.json()

caminho_bruto_municipios = os.path.join(DIR_BRUTOS_IBGE, "municipios_am_raw.json")
sha = salvar_json_bruto(caminho_bruto_municipios, municipios_am_raw)
registrar_provenancia(
    fonte="API IBGE - Localidades", url=URL_MUNICIPIOS_AM, metodo="GET", parametros={},
    observacao=f"{len(municipios_am_raw)} municípios", arquivo=caminho_bruto_municipios, sha256=sha,
)

if len(municipios_am_raw) != 62:
    print(f"[aviso] esperados 62 municípios do AM, vieram {len(municipios_am_raw)}.")
print(f"{len(municipios_am_raw)} municípios do AM coletados.")
pd.json_normalize(municipios_am_raw)[["id", "nome"]].head()

62 municípios do AM coletados.


,id,nome
0,1300029,Alvarães
1,1300060,Amaturá
2,1300086,Anamã
3,1300102,Anori
4,1300144,Apuí


In [ ]:
BASE_API_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"
_MISSING_SIDRA = {"", "-", "..", "...", "X"}


def obter_json(url, fonte, parametros=None, caminho_bruto=None):
    """GET + JSON + proveniência. Em caso de erro imprime o corpo da resposta (a API do IBGE
    costuma explicar o que está errado) e devolve None."""
    try:
        resp = http_get(url, timeout=30)
    except Exception as e:
        print(f"[ERRO de rede] {fonte}: {e}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros, observacao=f"ERRO: {e}")
        return None
    if resp.status_code != 200:
        print(f"[ERRO HTTP {resp.status_code}] {fonte}\n   corpo: {resp.text[:400]}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros,
                              observacao=f"ERRO HTTP {resp.status_code}: {resp.text[:200]}")
        return None
    try:
        dados = resp.json()
    except ValueError:
        print(f"[ERRO] {fonte}: resposta não é JSON: {resp.text[:200]}")
        registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros, observacao="ERRO: resposta não-JSON")
        return None
    sha = salvar_json_bruto(caminho_bruto, dados) if caminho_bruto else None
    registrar_provenancia(fonte=fonte, url=url, metodo="GET", parametros=parametros,
                          observacao="OK", arquivo=caminho_bruto, sha256=sha)
    return dados


def niveis_da_tabela(meta):
    """Conjunto de níveis territoriais (N1, N2, N3, N6, N7...) disponíveis na tabela.
    Aceita tanto {"Administrativo": ["N1","N3"], ...} quanto {"N3": true, ...}."""
    nt = meta.get("nivelTerritorial", {})
    niveis = set()
    if isinstance(nt, dict):
        for k, v in nt.items():
            if isinstance(v, (list, tuple, set)):
                niveis.update(v)
            elif v and re.fullmatch(r"N\d+", str(k)):
                niveis.add(str(k))
    return niveis


def escolher_variavel(variaveis, palavras_chave):
    """Escolhe a variável cujo nome contém alguma palavra-chave; senão, a primeira (com aviso)."""
    for v in variaveis:
        if any(p in v["nome"].lower() for p in palavras_chave):
            return v["id"]
    print(f"[aviso] nenhuma variável casou com {palavras_chave}; usando a primeira: {variaveis[0]['nome']}")
    return variaveis[0]["id"]


def montar_classificacao(meta, preferencias=("índice geral", "total", "geral")):
    """Muitas tabelas exigem `classificacao`. Para cada classificação escolhe a categoria mais
    agregada (por nome, na ordem de `preferencias`); se nenhuma casar, usa a primeira."""
    partes = []
    for classif in meta.get("classificacoes", []):
        cats = classif.get("categorias", [])
        if not cats:
            continue
        escolhida = None
        for pref in preferencias:
            escolhida = next((c for c in cats if pref in c["nome"].lower()), None)
            if escolhida:
                break
        escolhida = escolhida or cats[0]
        partes.append(f"{classif['id']}[{escolhida['id']}]")
    return "|".join(partes) if partes else None


def localizar_localidade(id_tabela, preferencias, niveis_disponiveis):
    """preferencias = [(nivel, trecho_do_nome), ...] em ordem de preferência.
    Consulta /agregados/{id}/localidades/{nivel} e devolve (nivel, id, nome) do 1º casamento."""
    cache = {}
    for nivel, trecho in preferencias:
        if nivel not in niveis_disponiveis:
            continue
        if nivel not in cache:
            cache[nivel] = obter_json(
                f"{BASE_API_AGREGADOS}/{id_tabela}/localidades/{nivel}",
                fonte=f"API IBGE - Localidades do agregado {id_tabela} ({nivel})",
                parametros={"nivel": nivel},
            ) or []
        for loc in cache[nivel]:
            if trecho in unidecode(loc["nome"]).lower():
                return nivel, str(loc["id"]), loc["nome"]
    return None


def inferir_frequencia(ids_periodos):
    """'trimestral' se os códigos AAAAQQ só usam QQ = 01..04; 'mensal' caso contrário
    (inclui trimestre móvel, em que QQ é o mês final, 01..12)."""
    sufixos = [int(p[4:]) for p in ids_periodos if len(p) == 6 and p.isdigit()]
    return "trimestral" if sufixos and max(sufixos) <= 4 else "mensal"


def periodo_para_meses(codigo, frequencia):
    """Converte um código de período do SIDRA em lista de 'AAAA-MM'.
    trimestral: 202304 (4º tri) -> ['2023-10','2023-11','2023-12'] | mensal: 202304 -> ['2023-04']."""
    codigo = str(codigo)
    if len(codigo) != 6 or not codigo.isdigit():
        return []
    ano, suf = int(codigo[:4]), int(codigo[4:])
    if frequencia == "trimestral":
        return [f"{ano}-{m:02d}" for m in range(3 * suf - 2, 3 * suf + 1)] if 1 <= suf <= 4 else []
    return [f"{ano}-{suf:02d}"] if 1 <= suf <= 12 else []


def periodos_na_janela(id_tabela):
    """Lista os períodos reais da tabela e devolve (ids_na_janela, frequencia)."""
    lista = obter_json(f"{BASE_API_AGREGADOS}/{id_tabela}/periodos",
                       fonte=f"API IBGE - Períodos do agregado {id_tabela}")
    if not lista:
        return [], None
    ids = [str(p["id"]) for p in lista]
    freq = inferir_frequencia(ids)
    janela = set(MESES_JANELA)
    return [i for i in ids if set(periodo_para_meses(i, freq)) & janela], freq


def montar_url_sidra(id_tabela, variavel, periodos, localidades, classificacao=None):
    url = (f"{BASE_API_AGREGADOS}/{id_tabela}/periodos/{'|'.join(periodos)}"
           f"/variaveis/{variavel}?localidades={localidades}")
    if classificacao:
        url += f"&classificacao={classificacao}"
    return url


def converter_valor_sidra(valor):
    s = str(valor).strip()
    if s in _MISSING_SIDRA:
        return None
    try:
        return float(s.replace(",", "."))
    except ValueError:
        return None


def sidra_para_df(sidra_json, nome_valor, frequencia):
    """Achata o JSON do SIDRA (lista de variáveis -> 'resultados' -> 'series' -> 'serie') em um
    DataFrame tidy [ano_mes, periodo_original, localidade, <nome_valor>]. Trimestres viram 3 meses."""
    colunas = ["ano_mes", "periodo_original", "localidade", nome_valor]
    linhas = []
    for variavel in sidra_json:
        for resultado in variavel.get("resultados", []):
            for serie in resultado.get("series", []):
                localidade = serie.get("localidade", {}).get("nome")
                for periodo, valor in serie.get("serie", {}).items():
                    v = converter_valor_sidra(valor)
                    for ano_mes in periodo_para_meses(periodo, frequencia):
                        linhas.append({"ano_mes": ano_mes, "periodo_original": str(periodo),
                                       "localidade": localidade, nome_valor: v})
    df = pd.DataFrame(linhas, columns=colunas)
    df[nome_valor] = df[nome_valor].astype(float)
    return df.drop_duplicates(subset="ano_mes", keep="last").reset_index(drop=True)


def coletar_indicador(id_tabela, rotulo, preferencias_localidade, palavras_variavel, arquivo_bruto):
    """Pipeline completo para um agregado: metadados -> variável -> classificação -> localidade ->
    períodos -> dados. Devolve dict com o JSON bruto e os parâmetros usados, ou None se falhar."""
    meta = obter_json(f"{BASE_API_AGREGADOS}/{id_tabela}/metadados",
                      fonte=f"API IBGE - Metadados do agregado {id_tabela}")
    if meta is None:
        return None

    print(f"[{rotulo}] Tabela {id_tabela}: {meta.get('nome')}")
    print("   Periodicidade:", meta.get("periodicidade"))
    print("   Variáveis:")
    for v in meta.get("variaveis", []):
        print("     ", v["id"], "-", v["nome"])
    niveis = niveis_da_tabela(meta)
    print("   Níveis territoriais:", sorted(niveis))

    variavel = escolher_variavel(meta["variaveis"], palavras_variavel)
    classificacao = montar_classificacao(meta)
    achado = localizar_localidade(id_tabela, preferencias_localidade, niveis)
    if achado is None:
        print(f"[{rotulo}] Nenhuma das localidades preferidas existe nesta tabela: {preferencias_localidade}")
        return None
    nivel, loc_id, loc_nome = achado
    periodos, frequencia = periodos_na_janela(id_tabela)
    if not periodos:
        print(f"[{rotulo}] Nenhum período da tabela intersecta a janela {JANELA_INICIO}..{JANELA_FIM}.")
        return None

    url = montar_url_sidra(id_tabela, variavel, periodos, f"{nivel}[{loc_id}]", classificacao)
    print(f"   Localidade: {loc_nome} ({nivel}[{loc_id}]) | frequência: {frequencia} | "
          f"{len(periodos)} períodos | classificação: {classificacao}")
    dados = obter_json(
        url, fonte=f"API IBGE - SIDRA ({rotulo})",
        parametros={"agregado": id_tabela, "variavel": variavel, "localidades": f"{nivel}[{loc_id}]",
                    "classificacao": classificacao, "periodos": f"{periodos[0]}..{periodos[-1]}"},
        caminho_bruto=arquivo_bruto,
    )
    if dados is None:
        return None
    return {"json": dados, "frequencia": frequencia, "localidade_nome": loc_nome, "url": url,
            "tabela_nome": meta.get("nome")}


print("Funções da API prontas.")

Funções da API prontas.


In [ ]:
# (Opcional, só para conferência) O ID pinado aparece no catálogo público do SIDRA? Com que nome?
catalogo = obter_json(BASE_API_AGREGADOS, fonte="API IBGE - Catálogo de Agregados (SIDRA)")
if catalogo:
    nomes_por_id = {str(a["id"]): a["nome"] for grupo in catalogo for a in grupo.get("agregados", [])}
    for rotulo, tid in [("IPCA", ID_TABELA_IPCA), ("Desocupação", ID_TABELA_DESOCUPACAO)]:
        print(f"{rotulo}: tabela {tid} -> {nomes_por_id.get(str(tid), '!! NÃO ENCONTRADA no catálogo !!')}")
else:
    print("Catálogo indisponível; a conferência será feita pelos metadados nas próximas células.")

IPCA: tabela 7060 -> IPCA - Variação mensal, acumulada no ano, acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços (a partir de janeiro/2020)
Desocupação: tabela 4099 -> Taxas de desocupação e de subutilização da força de trabalho, na semana de referência, das pessoas de 14 anos ou mais de idade


In [ ]:
# --- IPCA: Manaus não é área de coleta do IPCA -> procura Manaus; se não houver, usa proxy regional ---
res_ipca = coletar_indicador(
    ID_TABELA_IPCA, "IPCA",
    preferencias_localidade=[("N6", "manaus"), ("N7", "manaus"),      # caso o IBGE inclua Manaus no futuro
                             ("N7", "belem"), ("N6", "rio branco"),  # proxies regionais (Norte)
                             ("N1", "brasil")],                       # último recurso
    palavras_variavel=["variação mensal"],
    arquivo_bruto=os.path.join(DIR_BRUTOS_IBGE, "ipca_raw.json"),
)
if res_ipca is None:
    print("\nIPCA NÃO coletado. Veja a mensagem de erro acima (corpo da resposta da API) e ajuste ID_TABELA_IPCA.")
elif "manaus" not in unidecode(res_ipca["localidade_nome"]).lower():
    print(f"\n[ATENÇÃO] Manaus não existe na tabela {ID_TABELA_IPCA}. Usando PROXY: '{res_ipca['localidade_nome']}'."
          " Registre isso no dataset card (Seção 10, A.5).")

[IPCA] Tabela 7060: IPCA - Variação mensal, acumulada no ano, acumulada em 12 meses e peso mensal, para o índice geral, grupos, subgrupos, itens e subitens de produtos e serviços (a partir de janeiro/2020)
   Periodicidade: {'frequencia': 'mensal', 'inicio': 202001, 'fim': 202608}
   Variáveis:
      63 - IPCA - Variação mensal
      69 - IPCA - Variação acumulada no ano
      2265 - IPCA - Variação acumulada em 12 meses
      66 - IPCA - Peso mensal
   Níveis territoriais: ['N1', 'N6', 'N7']
   Localidade: Rio Branco (N6[1200401]) | frequência: mensal | 28 períodos | classificação: 315[7169]

[ATENÇÃO] Manaus não existe na tabela 7060. Usando PROXY: 'Rio Branco'. Registre isso no dataset card (Seção 10, A.5).


In [ ]:
# --- Taxa de desocupação (PNAD Contínua): Amazonas (UF); se faltar, Região Norte; senão Brasil ---
res_desoc = coletar_indicador(
    ID_TABELA_DESOCUPACAO, "Desocupação",
    preferencias_localidade=[("N3", "amazonas"), ("N2", "norte"), ("N1", "brasil")],
    palavras_variavel=["taxa de desocupação", "desocupação"],
    arquivo_bruto=os.path.join(DIR_BRUTOS_IBGE, "desocupacao_raw.json"),
)
if res_desoc is None:
    print("\nDesocupação NÃO coletada. Veja a mensagem de erro acima e confira ID_TABELA_DESOCUPACAO.")
else:
    if res_desoc["frequencia"] == "trimestral":
        print("\n[nota] Série TRIMESTRAL: cada trimestre será repetido nos seus 3 meses (Seção 6.2).")
    if "amazonas" not in unidecode(res_desoc["localidade_nome"]).lower():
        print(f"[ATENÇÃO] Usando '{res_desoc['localidade_nome']}' no lugar do Amazonas. Registre no dataset card.")

[Desocupação] Tabela 4099: Taxas de desocupação e de subutilização da força de trabalho, na semana de referência, das pessoas de 14 anos ou mais de idade
   Periodicidade: {'frequencia': 'trimestral', 'inicio': 201201, 'fim': 202602}
   Variáveis:
      4099 - Taxa de desocupação, na semana de referência, das pessoas de 14 anos ou mais de idade
      4103 - Coeficiente de variação - Taxa de desocupação, na semana de referência, das pessoas de 14 anos ou mais de idade
      4114 - Taxa combinada de desocupação e de subocupação por insuficiência de horas trabalhadas, na semana de referência, das pessoas de 14 anos ou mais de idade
      4115 - Coeficiente de variação - Taxa combinada de desocupação e de subocupação por insuficiência de horas trabalhadas, na semana de referência, das pessoas de 14 anos ou mais de idade
      4116 - Taxa combinada de desocupação e força de trabalho potencial, na semana de referência, das pessoas de 14 anos ou mais de idade
      4117 - Coeficiente de varia

## 5. Conferência da preservação do dado bruto (item 3.3)

In [ ]:
for pasta in [DIR_BRUTOS_SCRAPING, DIR_BRUTOS_IBGE]:
    print(f"\n{pasta}/")
    for arq in sorted(os.listdir(pasta)):
        caminho = os.path.join(pasta, arq)
        with open(caminho, "rb") as f:
            h = sha256_bytes(f.read())[:12]
        print(f"  - {arq}  ({os.path.getsize(caminho) / 1024:.1f} KB, sha256 {h}…)")


./dados_brutos/scraping_estiagem/
  - 2023-10-19_633364_amazonas_tem_a_menor_superficie_de_agua_desde_2018_mostra_mapbiomas.html  (64.0 KB, sha256 f0ab7130357b…)
  - 2023-10-23_seca_no_amazonas_afeta_mais_de_630_mil_pessoas.html  (83.8 KB, sha256 81177deaf3cd…)
  - 2023-10-25_633559_numero_de_afetados_pela_seca_severa_no_amazonas_sobe_para_633_mil_pess.html  (62.3 KB, sha256 68674c392ec3…)
  - 2023-10-27_seca_afeta_todos_os_62_municipios_do_amazonas_html.html  (122.3 KB, sha256 e6fbf4dc06b1…)
  - 2023-10-28_633708_quase_todos_os_municipios_do_am_estao_em_situacao_de_emergencia_por_causa.html  (60.6 KB, sha256 d18cf292eebd…)
  - 2024-07-12_641315_seca_no_amazonas_chega_antes_do_previsto_e_cidades_entram_em_emergencia.html  (62.8 KB, sha256 c96fb6dcb53c…)
  - 2024-08-23_642732_rios_amazonicos_registram_baixa_historica_um_mes_antes_do_pico_da_estiage.html  (62.3 KB, sha256 42bf5e0845cc…)
  - conferencia_manual_boletins.csv  (0.3 KB, sha256 d3f4df2400cf…)

./dados_brutos/api_ibge/
  - des

## 6. Tratamento, limpeza e integração das fontes

### 6.1 Série da estiagem (scraping + conferência manual)
Une o resultado automático (`df_scraping_raw`) com a conferência manual (`df_manual`) pela `data_ref`.
Para cada campo, **o valor manual tem prioridade** e a coluna `origem_<campo>` registra de onde veio
(`manual`, `automatico` ou vazio). Divergências entre os dois são listadas em `df_divergencias`
para o grupo decidir. O resultado é a base **por boletim** (`df_boletins`).

### 6.2 Indicadores econômicos (API)
O JSON do SIDRA é achatado por `sidra_para_df` (Seção 4). Séries **trimestrais** são expandidas para
os 3 meses do trimestre (o valor se repete), preservando o código original em `periodo_original`.

### 6.3 Integração (base mensal)
Chave de integração = `ano_mes`. Os boletins são **agregados por mês** (evita repetir o mesmo IPCA
em várias linhas e inflar artificialmente o n) e juntados a um **calendário completo** da janela de
análise. Assim, meses **sem boletim** continuam na base (com campos da estiagem vazios), mantendo o
contexto econômico de antes/depois da estiagem. Agregações por mês:
`municipios_emergencia`, `pessoas_afetadas_mil`, `familias_afetadas_mil` → **máximo**;
`cota_rio_negro_m` → **mínimo** (menor nível = pior situação); `n_boletins` → contagem.

> **Rótulo `em_crise`:** `True` se `municipios_emergencia >= LIMIAR_CRISE_MUNICIPIOS`; `False` se abaixo;
> **vazio (`<NA>`) quando não há boletim no mês** (ausência de dado ≠ ausência de crise).
> O limiar é arbitrário. Como `em_crise` é função direta de `municipios_emergencia`, **não use as duas
> juntas** em modelos futuros (vazamento de alvo).

Nomes de município (quando usados) são normalizados com `unidecode` + `str.upper().strip()`.

In [ ]:
# --- 6.1 Série da estiagem: manual tem prioridade; registra origem e divergências -------------
CAMPOS = ["municipios_emergencia", "pessoas_afetadas_mil", "familias_afetadas_mil", "cota_rio_negro_m"]
COLS_AUTO = ["data_ref", "fonte", "url", "titulo", "data_publicacao_meta", "data_divergente",
             "arquivo_bruto", "evidencias"] + CAMPOS

df_auto = df_scraping_raw.copy() if len(df_scraping_raw) else pd.DataFrame(columns=COLS_AUTO)
df_est = df_auto.merge(df_manual, on="data_ref", how="outer", suffixes=("_auto", "_manual"))

divergencias = []
for c in CAMPOS:
    manual, auto = df_est[f"{c}_manual"], df_est[f"{c}_auto"]
    df_est[c] = pd.to_numeric(manual.combine_first(auto), errors="coerce")   # manual tem prioridade
    df_est[f"origem_{c}"] = np.where(manual.notna(), "manual", np.where(auto.notna(), "automatico", None))
    dif = df_est[manual.notna() & auto.notna() & (manual != auto)]
    for _, r in dif.iterrows():
        divergencias.append({"data_ref": r["data_ref"], "fonte": r.get("fonte"), "campo": c,
                             "automatico": r[f"{c}_auto"], "manual": r[f"{c}_manual"]})

df_divergencias = pd.DataFrame(divergencias, columns=["data_ref", "fonte", "campo", "automatico", "manual"])
if len(df_divergencias):
    print(f"[atenção] {len(df_divergencias)} divergência(s) entre extração automática e conferência manual "
          "(vale o manual; confira qual está certo na página original):")
    display(df_divergencias)
else:
    print("Sem divergências entre automático e manual nos campos em que ambos existem.")

df_est["data_ref"] = pd.to_datetime(df_est["data_ref"])
df_est["ano_mes"] = df_est["data_ref"].dt.strftime("%Y-%m")

COLS_BOLETIM = (["data_ref", "ano_mes", "fonte", "url"] + CAMPOS + [f"origem_{c}" for c in CAMPOS]
                + ["data_publicacao_meta", "data_divergente"])
df_boletins = (df_est.reindex(columns=COLS_BOLETIM)
                     .sort_values(["data_ref", "fonte"]).reset_index(drop=True))
df_boletins

[atenção] 2 divergência(s) entre extração automática e conferência manual (vale o manual; confira qual está certo na página original):


,data_ref,fonte,campo,automatico,manual
0,2023-10-23,Jornal de Brasília,municipios_emergencia,62.0,59.0
1,2023-10-25,IHU/ClimaInfo,municipios_emergencia,62.0,59.0


,data_ref,ano_mes,fonte,url,municipios_emergencia,pessoas_afetadas_mil,familias_afetadas_mil,cota_rio_negro_m,origem_municipios_emergencia,origem_pessoas_afetadas_mil,origem_familias_afetadas_mil,origem_cota_rio_negro_m,data_publicacao_meta,data_divergente
0,2023-09-29,2023-09,NaN,NaN,55.0,NaN,NaN,NaN,manual,None,None,None,NaN,NaN
1,2023-10-19,2023-10,IHU/ClimaInfo,https://ihu.unisinos.br/633364-amazonas-tem-a-...,NaN,NaN,NaN,NaN,None,None,None,None,None,False
2,2023-10-23,2023-10,Jornal de Brasília,https://jornaldebrasilia.com.br/noticias/brasi...,59.0,630.0,158.0,12.89,manual,manual,automatico,manual,None,False
3,2023-10-25,2023-10,IHU/ClimaInfo,https://ihu.unisinos.br/633559-numero-de-afeta...,59.0,633.0,158.0,NaN,manual,manual,manual,None,None,False
4,2023-10-27,2023-10,Diário de Pernambuco,https://www.diariodepernambuco.com.br/noticia/...,62.0,608.0,152.0,12.70,manual,manual,manual,automatico,2025-09-02,True
5,2023-10-28,2023-10,IHU/ClimaInfo,https://ihu.unisinos.br/633708-quase-todos-os-...,60.0,600.0,NaN,NaN,automatico,automatico,None,None,None,False
6,2023-11-19,2023-11,NaN,NaN,62.0,598.0,150.0,NaN,manual,manual,manual,None,NaN,NaN
7,2024-07-12,2024-07,IHU/ClimaInfo,https://ihu.unisinos.br/641315-seca-no-amazona...,20.0,NaN,NaN,26.49,manual,None,None,automatico,None,False
8,2024-08-23,2024-08,IHU/ClimaInfo,https://ihu.unisinos.br/categorias/642732-rios...,53.0,NaN,NaN,NaN,manual,None,None,None,None,False


In [ ]:
# --- 6.2 Indicadores do IBGE em DataFrames tidy: ano_mes + valor -------------------------------
def _df_indicador(res, nome_valor, prefixo):
    colunas = ["ano_mes", nome_valor, f"{prefixo}_localidade", f"{prefixo}_periodo_original"]
    if res is None:
        print(f"[aviso] {nome_valor} não foi coletado na Seção 4 — seguindo com DataFrame vazio "
              "(a coluna ficará toda vazia na base).")
        return pd.DataFrame(columns=colunas)
    df = sidra_para_df(res["json"], nome_valor, res["frequencia"])
    return df.rename(columns={"localidade": f"{prefixo}_localidade",
                              "periodo_original": f"{prefixo}_periodo_original"})[colunas]


df_ipca = _df_indicador(res_ipca if "res_ipca" in globals() else None, "ipca_variacao_mensal", "ipca")
df_desoc = _df_indicador(res_desoc if "res_desoc" in globals() else None, "taxa_desocupacao", "desocupacao")

print("IPCA:", df_ipca.shape, "| Desocupação:", df_desoc.shape)
print("\nIPCA (amostra):")
display(df_ipca.head())
print("\nDesocupação (amostra — confira se cada trimestre aparece repetido nos 3 meses):")
display(df_desoc.head(6))

IPCA: (28, 4) | Desocupação: (30, 4)

IPCA (amostra):


,ano_mes,ipca_variacao_mensal,ipca_localidade,ipca_periodo_original
0,2022-09,-0.09,Rio Branco (AC),202209
1,2022-10,0.44,Rio Branco (AC),202210
2,2022-11,0.12,Rio Branco (AC),202211
3,2022-12,1.32,Rio Branco (AC),202212
4,2023-01,0.67,Rio Branco (AC),202301



Desocupação (amostra — confira se cada trimestre aparece repetido nos 3 meses):


,ano_mes,taxa_desocupacao,desocupacao_localidade,desocupacao_periodo_original
0,2022-07,9.4,Amazonas,202203
1,2022-08,9.4,Amazonas,202203
2,2022-09,9.4,Amazonas,202203
3,2022-10,10.1,Amazonas,202204
4,2022-11,10.1,Amazonas,202204
5,2022-12,10.1,Amazonas,202204


In [ ]:
# --- 6.3 Integração final (base mensal): calendário da janela + estiagem + IPCA + desocupação ----
agg_boletins = (df_boletins.groupby("ano_mes")
                .agg(n_boletins=("data_ref", "size"),
                     municipios_emergencia=("municipios_emergencia", "max"),
                     pessoas_afetadas_mil=("pessoas_afetadas_mil", "max"),
                     familias_afetadas_mil=("familias_afetadas_mil", "max"),
                     cota_rio_negro_m=("cota_rio_negro_m", "min"))
                .reset_index())

fora_da_janela = sorted(set(agg_boletins["ano_mes"]) - set(MESES_JANELA))
if fora_da_janela:
    print(f"[aviso] boletins em meses fora da janela (serão descartados da base mensal): {fora_da_janela}")

calendario = pd.DataFrame({"ano_mes": MESES_JANELA})
base_mensal = (calendario
               .merge(agg_boletins, on="ano_mes", how="left")
               .merge(df_ipca, on="ano_mes", how="left")
               .merge(df_desoc, on="ano_mes", how="left"))
base_mensal["n_boletins"] = base_mensal["n_boletins"].fillna(0).astype(int)

# Rótulo com dado ausente preservado (NÃO transformar NaN em "sem crise")
_m = base_mensal["municipios_emergencia"]
base_mensal["em_crise"] = (_m >= LIMIAR_CRISE_MUNICIPIOS).astype("boolean").mask(_m.isna())

# Dimensão de municípios do AM (código, nome, nome normalizado) — apoio para fases futuras.
# Ainda NÃO entra no merge: nenhuma das fontes atuais tem dado em nível de município.
df_municipios_am = pd.json_normalize(municipios_am_raw)[["id", "nome"]].rename(
    columns={"id": "municipio_id", "nome": "municipio_nome"})
df_municipios_am["municipio_nome_normalizado"] = df_municipios_am["municipio_nome"].apply(
    lambda s: unidecode(s).upper().strip())

base_mensal

,ano_mes,n_boletins,municipios_emergencia,pessoas_afetadas_mil,familias_afetadas_mil,cota_rio_negro_m,ipca_variacao_mensal,ipca_localidade,ipca_periodo_original,taxa_desocupacao,desocupacao_localidade,desocupacao_periodo_original,em_crise
0,2022-09,0,NaN,NaN,NaN,NaN,-0.09,Rio Branco (AC),202209,9.4,Amazonas,202203,<NA>
1,2022-10,0,NaN,NaN,NaN,NaN,0.44,Rio Branco (AC),202210,10.1,Amazonas,202204,<NA>
2,2022-11,0,NaN,NaN,NaN,NaN,0.12,Rio Branco (AC),202211,10.1,Amazonas,202204,<NA>
3,2022-12,0,NaN,NaN,NaN,NaN,1.32,Rio Branco (AC),202212,10.1,Amazonas,202204,<NA>
4,2023-01,0,NaN,NaN,NaN,NaN,0.67,Rio Branco (AC),202301,10.5,Amazonas,202301,<NA>
5,2023-02,0,NaN,NaN,NaN,NaN,0.44,Rio Branco (AC),202302,10.5,Amazonas,202301,<NA>
6,2023-03,0,NaN,NaN,NaN,NaN,0.54,Rio Branco (AC),202303,10.5,Amazonas,202301,<NA>
7,2023-04,0,NaN,NaN,NaN,NaN,0.64,Rio Branco (AC),202304,9.7,Amazonas,202302,<NA>
8,2023-05,0,NaN,NaN,NaN,NaN,0.29,Rio Branco (AC),202305,9.7,Amazonas,202302,<NA>
9,2023-06,0,NaN,NaN,NaN,NaN,-0.50,Rio Branco (AC),202306,9.7,Amazonas,202302,<NA>


In [ ]:
# --- Verificações de qualidade (rode e leia antes de salvar) --------------------------------
print("Dimensões:", base_mensal.shape, "| meses com boletim:", int((base_mensal["n_boletins"] > 0).sum()))
print("\n% de valores vazios por coluna:")
print((base_mensal.isna().mean() * 100).round(1).to_string())

alertas = []
if base_mensal["ipca_variacao_mensal"].isna().all():
    alertas.append("IPCA todo vazio: coleta falhou ou os períodos não intersectam a janela.")
if base_mensal["taxa_desocupacao"].isna().all():
    alertas.append("Desocupação toda vazia: coleta falhou ou os períodos não intersectam a janela.")
if base_mensal["n_boletins"].sum() == 0:
    alertas.append("Nenhum boletim na base mensal: scraping falhou/pulado e a conferência manual está fora da janela?")
if base_mensal["ano_mes"].duplicated().any():
    alertas.append("Há meses duplicados na base mensal (chave ano_mes deveria ser única).")
faltando = [m for m in ["2023-09", "2023-10", "2023-11"] if m not in set(agg_boletins["ano_mes"])]
if faltando:
    alertas.append(f"Sem boletim em meses centrais da estiagem de 2023: {faltando}.")

print("\nALERTAS:" if alertas else "\nSem alertas automáticos.")
for a in alertas:
    print(" -", a)

Dimensões: (28, 13) | meses com boletim: 5

% de valores vazios por coluna:
ano_mes                          0.0
n_boletins                       0.0
municipios_emergencia           82.1
pessoas_afetadas_mil            92.9
familias_afetadas_mil           92.9
cota_rio_negro_m                92.9
ipca_variacao_mensal             0.0
ipca_localidade                  0.0
ipca_periodo_original            0.0
taxa_desocupacao                 0.0
desocupacao_localidade           0.0
desocupacao_periodo_original     0.0
em_crise                        82.1

Sem alertas automáticos.


> **Nota de limitação (registrar no *dataset card*, item A.5):** a série de scraping tem poucos
> pontos (5 meses com boletim na janela, com um buraco entre nov/2023 e jul/2024), enquanto as
> séries do IBGE são contínuas; e a coleta termina em ago/2024, **antes do pico da estiagem de
> 2024**. Para as fases futuras (EDA, séries temporais), o grupo deve ampliar a coleta —
> idealmente com uma fonte estruturada (ex.: cota diária do Rio Negro em Manaus pelo SGB/CPRM ou
> ANA/Hidroweb) em vez de matérias de imprensa esparsas. Isso está documentado como *lacuna conhecida*.

## 7. Base tratada — salvar em CSV e Parquet

In [ ]:
caminho_csv = os.path.join(DIR_TRATADOS, "base_estiagem_indicadores_am.csv")
caminho_parquet = os.path.join(DIR_TRATADOS, "base_estiagem_indicadores_am.parquet")
caminho_boletins = os.path.join(DIR_TRATADOS, "base_boletins_estiagem.csv")
caminho_municipios_tratado = os.path.join(DIR_TRATADOS, "municipios_am.csv")

base_mensal.to_csv(caminho_csv, index=False)
try:
    base_mensal.to_parquet(caminho_parquet, index=False)
except Exception as e:
    print("Não foi possível salvar em Parquet (instale 'pyarrow'):", e)
df_boletins.to_csv(caminho_boletins, index=False)
df_municipios_am.to_csv(caminho_municipios_tratado, index=False)

print("Salvos em:")
for c in [caminho_csv, caminho_parquet, caminho_boletins, caminho_municipios_tratado]:
    print(" ", c)
print("\nBase mensal (principal):", base_mensal.shape, "| Base por boletim:", df_boletins.shape)

Salvos em:
  ./dados_tratados/base_estiagem_indicadores_am.csv
  ./dados_tratados/base_estiagem_indicadores_am.parquet
  ./dados_tratados/base_boletins_estiagem.csv
  ./dados_tratados/municipios_am.csv

Base mensal (principal): (28, 13) | Base por boletim: (9, 14)


## 8. Registro de proveniência (item 3.4) — salvar log completo e empacotar

In [ ]:
df_provenance = pd.DataFrame(provenance_log)
caminho_provenance = os.path.join(BASE_DIR, "registro_proveniencia.csv")
df_provenance.to_csv(caminho_provenance, index=False)
print(f"{len(df_provenance)} eventos registrados em {caminho_provenance}")
df_provenance

26 eventos registrados em ./registro_proveniencia.csv


,fonte,url,data_hora_coleta_utc,metodo,parametros,observacao,arquivo_bruto,sha256
0,robots.txt de www.poder360.com.br,https://www.poder360.com.br/robots.txt,2026-09-19T19:46:29+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
1,robots.txt de ihu.unisinos.br,https://ihu.unisinos.br/robots.txt,2026-09-19T19:46:30+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
2,robots.txt de jornaldebrasilia.com.br,https://jornaldebrasilia.com.br/robots.txt,2026-09-19T19:46:31+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
3,robots.txt de www.diariodepernambuco.com.br,https://www.diariodepernambuco.com.br/robots.txt,2026-09-19T19:46:31+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
4,robots.txt de redeondadigital.com.br,https://redeondadigital.com.br/robots.txt,2026-09-19T19:46:31+00:00,GET (requests + urllib.robotparser),"{""user_agent"": ""TesteAcademico""}",status=ok; HTTP 200,None,None
5,Poder360,https://www.poder360.com.br/brasil/governo-do-...,2026-09-19T19:46:31+00:00,GET (requests),{},ERRO: 403 Client Error: Forbidden for url: htt...,None,None
6,IHU/ClimaInfo,https://ihu.unisinos.br/633364-amazonas-tem-a-...,2026-09-19T19:46:35+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_estiagem/2023-10-19_63...,f0ab7130357b5175d35c06b37c8598a91eb014b725065d...
7,Jornal de Brasília,https://jornaldebrasilia.com.br/noticias/brasi...,2026-09-19T19:46:37+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_estiagem/2023-10-23_se...,81177deaf3cd2d18f78161851479285c790c5a85e06ca5...
8,IHU/ClimaInfo,https://ihu.unisinos.br/633559-numero-de-afeta...,2026-09-19T19:46:41+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_estiagem/2023-10-25_63...,68674c392ec32dca192fdc12f2d72c47026fdd3ad4753f...
9,Diário de Pernambuco,https://www.diariodepernambuco.com.br/noticia/...,2026-09-19T19:46:43+00:00,GET (requests + BeautifulSoup),"{""headers"": ""User-Agent customizado"", ""timeout...",HTTP 200,./dados_brutos/scraping_estiagem/2023-10-27_se...,e6fbf4dc06b13625186daa87d5098e2a8d14b987df901f...


In [ ]:
# Empacota a entrega. O Colab apaga o disco ao desconectar: baixe o zip (ou use MONTAR_DRIVE=True).
# HTML bruto de matérias é conteúdo protegido por direitos autorais: por padrão fica FORA do zip.
caminho_zip = os.path.join(BASE_DIR, "entrega_trabalho1.zip")
with zipfile.ZipFile(caminho_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(caminho_provenance, "registro_proveniencia.csv")
    for pasta in [DIR_TRATADOS, DIR_BRUTOS]:
        for raiz, _, arquivos in os.walk(pasta):
            for a in arquivos:
                if a.endswith(".html") and not INCLUIR_HTML_BRUTO_NO_ZIP:
                    continue
                caminho = os.path.join(raiz, a)
                z.write(caminho, os.path.relpath(caminho, BASE_DIR))
print("Zip criado:", caminho_zip, f"({os.path.getsize(caminho_zip) / 1024:.0f} KB)")

try:
    from google.colab import files
    files.download(caminho_zip)
except ImportError:
    print("(fora do Colab: pegue o zip no caminho acima)")

Zip criado: ./entrega_trabalho1.zip (12 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Postura ética e legal (item 3.5)

- **robots.txt:** verificado programaticamente para **cada URL** (Seção 2). O resultado desta
  execução está em `df_robots`; o resumo é impresso na célula abaixo — **copie-o para o dataset
  card (A.6)**. URLs `bloqueado`/`indeterminado` foram puladas (salvo decisão explícita do grupo).
- **Licença / termos de uso de cada fonte:**
  - *Portais de notícia* (Poder360, IHU/Unisinos, Jornal de Brasília, Diário de Pernambuco, Rede
    Onda Digital): conteúdo com direitos autorais jornalísticos. **Não reproduzimos o texto das
    matérias** na base tratada — extraímos apenas **fatos numéricos** (nº de municípios, pessoas
    afetadas, cota do rio), com o link da fonte para atribuição. O HTML bruto é guardado apenas
    para auditoria/reprodutibilidade e **não deve ser redistribuído** (repositório público, entrega
    com HTMLs): por isso fica fora do zip por padrão. Os trechos curtos em `evidencias`
    (`df_scraping_raw`) servem à conferência interna — se a base for publicada, remova essa coluna.
  - *API do IBGE* (Localidades e SIDRA): dados públicos abertos, disponibilizados pelo próprio órgão
    federal para reuso, inclusive acadêmico ([Portal de Serviços do IBGE](https://servicodados.ibge.gov.br/api/docs)).
- **Dados pessoais / LGPD:** a base não contém dados pessoais identificáveis — os números
  ("pessoas afetadas") são agregados populacionais. Não se aplica minimização/anonimização adicional.
- **Não sobrecarregar servidores:** delay de `DELAY_ENTRE_REQUISICOES` (ou o `Crawl-delay` do site,
  se maior) entre requisições de scraping, sem paralelismo; retries limitados (2–3 tentativas com
  espera crescente). O total de requisições fica no log de proveniência.

In [ ]:
# Resumo para copiar no dataset card (A.6) — gerado a partir do que realmente aconteceu nesta execução
print("robots.txt — resultado por URL:")
print(df_robots["resultado"].value_counts().to_string())
puladas = df_robots[df_robots["resultado"] != "permitido"]
if len(puladas):
    print("\nURLs NÃO raspadas (bloqueadas/indeterminadas):")
    for _, r in puladas.iterrows():
        print(f" - {r['url']}  [{r['resultado']}; {r['detalhe']}]")
print("\nRequisições registradas no log:", len(provenance_log))

robots.txt — resultado por URL:
resultado
permitido    9

Requisições registradas no log: 26


## 10. Dataset Card (Apêndice A)

### A.1 Identificação
- **Nome da base:** Estiagem AM 2023-2024 × Indicadores Econômicos

### A.2 Fontes e proveniência
- **Fonte 1 — nome e URLs:** boletins/matérias sobre a estiagem 2023-2024 (ver `URLS_ESTIAGEM`:
  Poder360, IHU/Unisinos, Jornal de Brasília, Diário de Pernambuco e Rede Onda Digital).
- **Fonte 1 — método:** Web scraping (`requests` + `BeautifulSoup`), regex com validação de faixa
  + conferência manual (que tem prioridade; origem de cada valor em `origem_*`).
- **Fonte 1 — licença/termos:** conteúdo jornalístico; usamos apenas fatos numéricos, com atribuição de URL.
- **Fonte 2 — nome e URL:** API do IBGE — Localidades (`servicodados.ibge.gov.br/api/v1/localidades`)
  e SIDRA/Agregados (`servicodados.ibge.gov.br/api/v3/agregados`): tabela 7060 (IPCA) e tabela 4099
  (PNAD Contínua trimestral — taxa de desocupação).
- **Fonte 2 — método:** API REST (JSON).
- **Fonte 2 — licença/termos:** dados públicos abertos do IBGE.
- **Chave de integração:** `ano_mes` (data do boletim arredondada para o mês, casada com a
  granularidade mensal do IPCA e com os meses de cada trimestre da PNAD).

### A.3 Dicionário de variáveis (`base_estiagem_indicadores_am.csv` — base mensal)

| Variável | Tipo | Descrição | Unidade |
|---|---|---|---|
| ano_mes | categórica | Mês de referência (chave; um registro por mês da janela) | AAAA-MM |
| n_boletins | numérica discreta | Nº de boletins/matérias coletados no mês | contagem |
| municipios_emergencia | numérica discreta | Nº de municípios do AM em situação de emergência (máx. do mês) | contagem (0-62) |
| pessoas_afetadas_mil | numérica contínua | Pessoas afetadas pela estiagem (máx. do mês) | milhares de pessoas |
| familias_afetadas_mil | numérica contínua | Famílias afetadas pela estiagem (máx. do mês) | milhares de famílias |
| cota_rio_negro_m | numérica contínua | Nível do Rio Negro em Manaus (mín. do mês) | metros |
| ipca_variacao_mensal | numérica contínua | Variação mensal do IPCA na localidade de `ipca_localidade` | % |
| ipca_localidade | categórica | Localidade efetivamente usada no IPCA (**proxy regional**: Manaus não é área do IPCA) | - |
| ipca_periodo_original | categórica | Código do período no SIDRA | AAAAMM |
| taxa_desocupacao | numérica contínua | Taxa de desocupação (PNAD Contínua; **trimestral repetida nos 3 meses**) | % |
| desocupacao_localidade | categórica | Localidade da desocupação (AM/UF ou fallback) | - |
| desocupacao_periodo_original | categórica | Trimestre original no SIDRA (AAAA0T) | AAAA01–AAAA04 |
| em_crise | booleana anulável | `True` se `municipios_emergencia >= 50`; `<NA>` se não há boletim no mês | - |

`base_boletins_estiagem.csv` (granularidade de boletim): `data_ref`, `ano_mes`, `fonte`, `url`, os 4
campos numéricos, `origem_<campo>` (manual/automatico), `data_publicacao_meta`, `data_divergente`.
`municipios_am.csv`: `municipio_id`, `municipio_nome`, `municipio_nome_normalizado`.

### A.4 Volume e granularidade
- **Nº de linhas / colunas:** ver a célula de resumo logo abaixo (preencher aqui após rodar).
- **O que representa uma linha:** um **mês** da janela de análise, com o resumo dos boletins do mês
  (quando houver) e os indicadores econômicos do mês.
- **Cobertura:** Amazonas (estado) para a estiagem; IPCA de proxy regional e PNAD do AM (ver colunas
  `*_localidade`); janela `2022-09` a `2024-12` (boletins apenas em set–nov/2023 e jul–ago/2024).

### A.5 Limitações e decisões
- **Dados descartados:** URLs puladas por `robots.txt` (ver Seção 9); valores extraídos fora da
  faixa plausível são descartados; boletins fora da janela não entram na base mensal.
- **Lacunas conhecidas:** série de scraping esparsa e com republicações; buraco entre nov/2023 e
  jul/2024; coleta termina antes do pico da estiagem de 2024; **IPCA não cobre Manaus** (proxy);
  desocupação trimestral repetida por mês; conferência manual a ser revalidada pelo grupo.
- **Decisões de limpeza relevantes:** manual > automático (com `origem_*`); agregação mensal
  (máx./mín.); `em_crise` com limiar arbitrário e sem transformar ausência em `False`;
  nomes de município normalizados com `unidecode`.

### A.6 Considerações éticas
- **Contém dados pessoais?** Não.
- **Restrições de uso/redistribuição:** ver Seção 9 (não redistribuir o HTML/texto das matérias;
  dados do IBGE são abertos).
- **robots.txt verificado?** Sim, por URL — copiar o resumo da célula da Seção 9.

In [ ]:
# Resumo para preencher o A.4 do dataset card
print("Base mensal:", base_mensal.shape[0], "linhas x", base_mensal.shape[1], "colunas")
print("Base por boletim:", df_boletins.shape[0], "linhas x", df_boletins.shape[1], "colunas")
print("Período:", base_mensal["ano_mes"].min(), "a", base_mensal["ano_mes"].max())
print("Meses com boletim:", ", ".join(base_mensal.loc[base_mensal["n_boletins"] > 0, "ano_mes"]))

Base mensal: 28 linhas x 13 colunas
Base por boletim: 9 linhas x 14 colunas
Período: 2022-09 a 2024-12
Meses com boletim: 2023-09, 2023-10, 2023-11, 2024-07, 2024-08


Limitações Conhecidas

A arquitetura das fontes utilizadas e os métodos de agregação aplicados introduzem restrições que exigem cautela ao interpretar análises estatísticas ou projetar modelos preditivos futuros.

1. Limitações Metodológicas

Viés de Amostragem na Agregação Climática: A extração de variáveis meteorológicas (como temperatura máxima, umidade mínima e focos de calor) baseia-se em uma amostra reduzida e esparsa de boletins informativos, frequentemente limitados a dois ou três dias por mês. Consequentemente, a aplicação de funções de agregação (como max ou min) pode refletir anomalias pontuais e distorcer o verdadeiro perfil climático mensal consolidado.

Fragilidade na Extração de Textos Não Estruturados: A coleta de métricas numéricas a partir de narrativas jornalísticas por meio de expressões regulares (RegEx) permanece vulnerável a ruídos semânticos, mesmo com camadas de validação. O algoritmo pode capturar inadvertidamente valores de sensação térmica, previsões futuras ou dados de cidades adjacentes mencionadas no texto, confundindo-os com as medições efetivas da localidade-alvo.

Distorção de Granularidade Temporal (PNAD): O cruzamento de registros de periodicidade mensal com indicadores econômicos trimestrais (como a PNAD Contínua) exige a replicação artificial dos dados para preencher o trimestre. Esta abordagem gera patamares estáticos desprovidos de variação intra-trimestral, mascarando as flutuações reais e orgânicas do mercado de trabalho mês a mês e reduzindo a precisão de modelos de correlação temporal.